# Scaling Laws: 모델 크기의 법칙 - 실습 코드 1: Scaling Laws 실험: 파라미터 수 vs Loss 시뮬레이션

- Tutorial ID: `expand-scaling-laws`
- Tutorial: Scaling Laws: 모델 크기의 법칙
- Section ID: `expand-scaling-laws-code-1`
- Section: 실습 코드 1: Scaling Laws 실험: 파라미터 수 vs Loss 시뮬레이션


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 1: Scaling Laws 실험: 파라미터 수 vs Loss 시뮬레이션
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) 입력 데이터가 어떤 중간 변수들을 거쳐 최종 출력으로 변환되는지 shape 중심으로 추적
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ── Kaplan Scaling Laws 재현 ──
# L(N) = (N_c / N)^alpha_N
# L(D) = (D_c / D)^alpha_D

# 논문의 추정 상수
N_c = 8.8e13   # 파라미터 상수
D_c = 5.4e13   # 데이터 상수
alpha_N = 0.076
alpha_D = 0.095

# 모델 크기 범위
param_counts = np.logspace(7, 11, 50)  # 10M ~ 100B
data_counts = np.logspace(9, 13, 50)   # 1B ~ 10T tokens

# Loss 계산
loss_by_params = (N_c / param_counts) ** alpha_N
loss_by_data = (D_c / data_counts) ** alpha_D

print("=== Scaling Laws: 모델 크기별 예측 Loss ===")
for n, l in zip([1e7, 1e8, 1e9, 1e10, 1e11], loss_by_params[::10]):
    print(f"  {n:.0e} params → Loss: {l:.4f}")

# ── Chinchilla 최적 분석 ──
def chinchilla_optimal(compute_budget):
    """주어진 연산량(FLOPs)에서 최적의 N과 D 계산"""
    # C = 6 * N * D (Transformer 근사)
    # N_opt ∝ C^0.50, D_opt ∝ C^0.50
        A = 0.060  # Chinchilla 논문 상수
        B = 0.070
        alpha = 0.34
    beta = 0.28
    
    N_opt = (A * compute_budget / (B * 6)) ** (beta / (alpha + beta))
    D_opt = (B * compute_budget / (A * 6)) ** (alpha / (alpha + beta))
    
    return N_opt, D_opt

print("\n=== Chinchilla 최적: Compute별 최적 모델/데이터 크기 ===")
for compute in [1e20, 1e21, 1e22, 1e23, 1e24]:
    n_opt, d_opt = chinchilla_optimal(compute)
    print(f"  C={compute:.0e} FLOPs → N_opt={n_opt:.2e}, D_opt={d_opt:.2e}")
    print(f"    ({n_opt/1e9:.1f}B params, {d_opt/1e9:.0f}B tokens)")

# ── 과소 학습 분석 ──
print("\n=== 과소 학습 분석 ===")
models = {
    "GPT-3 175B": (175e9, 300e9),
    "Gopher 280B": (280e9, 300e9),
    "Chinchilla 70B": (70e9, 1400e9),
}
for name, (n, d) in models.items():
    compute = 6 * n * d
    n_opt, d_opt = chinchilla_optimal(compute)
    ratio = d / (20 * n)  # Chinchilla 최적: tokens ≈ 20 × params
    print(f"  {name}: {n/1e9:.0f}B params, {d/1e9:.0f}B tokens")
    print(f"    Chinchilla 최적: {n_opt/1e9:.0f}B params, {d_opt/1e9:.0f}B tokens")
    print(f"    데이터/파라미터 비율: {d/n:.0f}x (최적: 20x)")